In [61]:
import os
import operator
from typing import TypedDict, Annotated,Literal,Type
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import google.genai as genai
from google.genai import types
from langgraph.graph import StateGraph, START, END


In [62]:
load_dotenv()
api_key = os.environ.get("GOOGLE_API_KEY")
if not api_key:
    raise RuntimeError("Please set GOOGLE_API_KEY in your environment or .env file.")
client = genai.Client(api_key=api_key)
MODEL_ID = 'gemini-2.5-flash'

In [63]:
class EvaluationSchema2(BaseModel):
    sentiment: str = Field(description='Detailed')

In [64]:
class EvaluationSchema(BaseModel):
    sentiment: Literal["positive","negative"] = Field(description='sentiment of the review')

In [65]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')

In [66]:
def get_structured_evaluation(prompt: str, schema: Type[BaseModel]) -> BaseModel:
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=schema,  # <--- Yaha dynamic schema use hoga
        ),
    )
    return schema.model_validate_json(response.text)

In [67]:
# def get_structured_evaluation(prompt: str) -> EvaluationSchema:
#     response = client.models.generate_content(
#         model=MODEL_ID,
#         contents=prompt,
#         config=types.GenerateContentConfig(
#             response_mime_type="application/json",
#             response_schema=EvaluationSchema,
#         ),
#     )
#     # Parse the valid JSON text directly back into your Pydantic schema
#     return EvaluationSchema.model_validate_json(response.text)


In [68]:
# def get_structured_evaluation2(prompt: str) -> EvaluationSchema:
#     response = client.models.generate_content(
#         model=MODEL_ID,
#         contents=prompt,
#         config=types.GenerateContentConfig(
#             response_mime_type="application/json",
#             response_schema=DiagnosisSchema,
#         ),
#     )
#     # Parse the valid JSON text directly back into your Pydantic schema
#     return DiagnosisSchema.model_validate_json(response.text)

In [69]:
# prompt = 'What is the sentiment of the following review - The software too good'
# get_structured_evaluation(prompt)

In [70]:
class ReviewState(TypedDict):
    review:str
    sentiment:Literal["positive","negative"]
    diagnosis:dict
    response:str

In [71]:
def find_sentiment(state: ReviewState):

    prompt = f'For the following review find out the sentiment \n {state["review"]}'
    sentiment = get_structured_evaluation(prompt,EvaluationSchema).sentiment

    return {'sentiment': sentiment}

def check_sentiment(state: ReviewState) -> Literal["positive_response", "run_diagnosis"]:

    if state['sentiment'] == 'positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'
    
def positive_response(state: ReviewState):

    prompt = f"""Write a warm thank-you message in response to this review:
    \n\n\"{state['review']}\"\n
Also, kindly ask the user to leave feedback on our website."""
    
    response = get_structured_evaluation(prompt,EvaluationSchema2)

    return {'response': response}

def run_diagnosis(state: ReviewState):

    prompt = f"""Diagnose this negative review:\n\n{state['review']}\n"
    "Return issue_type, tone, and urgency.
"""
    response = get_structured_evaluation(prompt,DiagnosisSchema)

    return {'diagnosis': response.model_dump()}

def negative_response(state: ReviewState):

    diagnosis = state['diagnosis']

    prompt = f"""You are a support assistant.
The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
Write an empathetic, helpful resolution message.
"""
    response = get_structured_evaluation(prompt,EvaluationSchema2)

    return {'response': response}

In [72]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment', find_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)

graph.add_edge(START, 'find_sentiment')

graph.add_conditional_edges('find_sentiment', check_sentiment)

graph.add_edge('positive_response', END)

graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)

workflow = graph.compile()

In [73]:
intial_state={
    'review': "I’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality."
}
workflow.invoke(intial_state)

{'review': 'I’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality.',
 'sentiment': 'negative',
 'diagnosis': {'issue_type': 'Bug', 'tone': 'frustrated', 'urgency': 'high'},
 'response': EvaluationSchema2(sentiment="I completely understand how frustrating it is to encounter a bug, especially when it's marked with high urgency. Please accept my sincerest apologies for the inconvenience this has caused. Our team is already actively investigating this issue as a top priority. We're working diligently to identify the root cause and implement a fix as quickly as possible. We will keep you updated on our progress and notify you as soon as a resolution is available. Thank you for your patience and understanding.")}